# Sales pipeline lakehouse

Loads raw CRM-style CSV files into a Fabric Lakehouse, cleans them and builds a star schema for Power BI.

1. Bronze - raw files copied into Delta tables
2. Silver - cleaned data (types, duplicates, bad rows)
3. Gold - dim / fact tables for the report

Before running: upload the 5 CSV files from `data/raw` into the Lakehouse folder `Files/raw` and attach the Lakehouse to this notebook.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SNAPSHOT_DATE = "2025-06-30"   # date the raw data was extracted


# small helper so every table is written the same way
def save(df, table_name):
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table_name)

## 1. Bronze - load the raw files

In [ ]:
files = {
    "sellers": "sellers.csv",
    "accounts": "accounts.csv",
    "opportunities": "opportunities.csv",
    "stage_history": "opportunity_stage_history.csv",
    "activities": "activities.csv",
}

for table, file in files.items():
    df = spark.read.option("header", True).csv("Files/raw/" + file)   # everything is text at this point
    df = df.withColumn("load_ts", F.current_timestamp())
    save(df, "bronze_" + table)
    print(table, df.count())

## 2. Silver - clean the data

In [ ]:
sellers = (spark.table("bronze_sellers").drop("load_ts")
    .withColumn("hire_date", F.to_date("hire_date"))
    .withColumn("quota_annual", F.col("quota_annual").cast("double"))
    .withColumn("is_active", F.col("is_active") == "True")
    .dropDuplicates(["seller_id"]))

accounts = (spark.table("bronze_accounts").drop("load_ts")
    .withColumn("created_date", F.to_date("created_date"))
    .dropDuplicates(["account_id"]))

save(sellers, "silver_sellers")
save(accounts, "silver_accounts")

Opportunities need the most work: spaces and casing in `stage`, lower-case currency codes, two different date formats,
duplicated rows, and a few rows that are simply wrong (negative amount, account that does not exist).

In [ ]:
opp = spark.table("bronze_opportunities").drop("load_ts")

# text clean-up
opp = (opp
    .withColumn("stage", F.initcap(F.regexp_replace(F.trim("stage"), " +", " ")))
    .withColumn("currency", F.upper(F.trim("currency"))))

# data types - expected_close_date comes in two formats (2024-03-15 and 15/03/2024)
opp = (opp
    .withColumn("probability", F.col("probability").cast("double"))
    .withColumn("amount", F.col("amount").cast("double"))
    .withColumn("created_date", F.to_date("created_date"))
    .withColumn("expected_close_date", F.coalesce(F.to_date("expected_close_date", "yyyy-MM-dd"),
                                                  F.to_date("expected_close_date", "dd/MM/yyyy")))
    .withColumn("actual_close_date", F.to_date("actual_close_date"))
    .withColumn("last_modified_ts", F.to_timestamp("last_modified_ts")))

# duplicates - drop exact copies, then keep only the latest version of each opportunity
latest = Window.partitionBy("opportunity_id").orderBy(F.col("last_modified_ts").desc())
opp = (opp.dropDuplicates()
    .withColumn("rn", F.row_number().over(latest))
    .filter("rn = 1")
    .drop("rn"))

In [ ]:
# rows we cannot trust go to a quarantine table instead of the star schema
known_accounts = accounts.select("account_id").withColumn("account_found", F.lit(True))

opp = (opp.join(known_accounts, "account_id", "left")
    .withColumn("dq_issue",
        F.when(F.col("account_found").isNull(), "account does not exist")
         .when(F.col("amount") < 0, "negative amount")))

quarantine = opp.filter("dq_issue is not null").drop("account_found")
opp = opp.filter("dq_issue is null").drop("dq_issue", "account_found")

save(quarantine, "silver_opportunities_quarantine")
save(opp, "silver_opportunities")

print("clean opportunities:", opp.count(), "| quarantined:", quarantine.count())

In [ ]:
opp_dates = opp.select("opportunity_id", "created_date")

history = (spark.table("bronze_stage_history").drop("load_ts")
    .withColumn("changed_at", F.to_timestamp("changed_at"))
    .dropDuplicates(["history_id"])
    .join(opp_dates.select("opportunity_id"), "opportunity_id", "left_semi"))   # only known opportunities

activities = (spark.table("bronze_activities").drop("load_ts")
    .withColumn("activity_ts", F.to_timestamp("activity_ts"))
    .withColumn("duration_minutes", F.col("duration_minutes").cast("int"))
    .dropDuplicates(["activity_id"])
    .join(opp_dates, "opportunity_id")
    .filter("activity_ts >= created_date")   # an activity cannot happen before the deal existed
    .drop("created_date"))

save(history, "silver_stage_history")
save(activities, "silver_activities")

## 3. Gold - star schema for Power BI

In [ ]:
dim_date = (spark.sql("SELECT explode(sequence(to_date('2018-01-01'), to_date('2026-12-31'), interval 1 day)) AS date")
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("day_name", F.date_format("date", "EEE"))
    # fiscal year starts in July: July 2024 belongs to FY2025
    .withColumn("fiscal_year", F.when(F.month("date") >= 7, F.year("date") + 1).otherwise(F.year("date")))
    .withColumn("fiscal_quarter", (F.floor(((F.month("date") + 5) % 12) / 3) + 1).cast("int")))

dim_stage = spark.createDataFrame(
    [("Prospecting", 1), ("Qualification", 2), ("Proposal", 3),
     ("Negotiation", 4), ("Closed Won", 5), ("Closed Lost", 6)],
    ["stage", "stage_order"])

save(dim_date, "dim_date")
save(dim_stage, "dim_stage")
save(spark.table("silver_sellers"), "dim_seller")
save(spark.table("silver_accounts"), "dim_account")

In [ ]:
def date_key(col):
    return F.date_format(col, "yyyyMMdd").cast("int")

snapshot = F.to_date(F.lit(SNAPSHOT_DATE))

fact_opportunity = (spark.table("silver_opportunities")
    .withColumn("created_date_key", date_key("created_date"))
    .withColumn("expected_close_date_key", date_key("expected_close_date"))
    .withColumn("actual_close_date_key", date_key("actual_close_date"))
    .withColumn("is_closed", F.col("stage").isin("Closed Won", "Closed Lost"))
    .withColumn("is_won", F.col("stage") == "Closed Won")
    .withColumn("weighted_amount", F.col("amount") * F.col("probability"))
    .withColumn("days_to_close", F.datediff("actual_close_date", "created_date"))
    .withColumn("days_open", F.when(~F.col("is_closed"), F.datediff(snapshot, "created_date"))))

# how long a deal stayed in each stage (the current stage runs until the snapshot date)
next_change = Window.partitionBy("opportunity_id").orderBy("changed_at")

fact_stage_history = (spark.table("silver_stage_history")
    .withColumn("changed_date_key", date_key("changed_at"))
    .withColumn("left_stage_at", F.lead("changed_at").over(next_change))
    .withColumn("days_in_stage", F.datediff(F.coalesce("left_stage_at", snapshot), "changed_at")))

fact_activity = (spark.table("silver_activities")
    .withColumn("activity_date_key", date_key("activity_ts")))

save(fact_opportunity, "fact_opportunity")
save(fact_stage_history, "fact_stage_history")
save(fact_activity, "fact_activity")

## 4. Quick check

In [ ]:
%%sql
SELECT s.stage_order, f.stage, COUNT(*) AS deals, ROUND(SUM(f.amount)) AS amount
FROM fact_opportunity f
JOIN dim_stage s ON s.stage = f.stage
GROUP BY s.stage_order, f.stage
ORDER BY s.stage_order

In [ ]:
%%sql
SELECT dq_issue, COUNT(*) AS rows_quarantined
FROM silver_opportunities_quarantine
GROUP BY dq_issue